In [ ]:
# ==============================================================================
# PROOF OF CONCEPT: Applying LoRA (Low-Rank Adaptation) to a Vision-Language Model
# ==============================================================================
import torch
from transformers import VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model

def setup_lora_model(model_name="naver-clova-ix/donut-base"):
    """
    Wraps a heavy VLM with LoRA adapters so we can finetune it on custom 
    passport/visa datasets using 1/10th of the standard VRAM footprint.
    """
    print(f"Loading base model: {model_name}")
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    
    # Define the LoRA configuration targeting the attention matrices
    config = LoraConfig(
        r=16,                     # Rank of the update matrices
        lora_alpha=32,            # Scaling factor
        target_modules=["query", "value"], # Standard practice for Transformer attention blocks
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"     # The Donut decoder is autoregressive
    )
    
    # Inject the LoRA adapters into the base model
    lora_model = get_peft_model(model, config)
    
    # Print the trainable parameter reduction
    lora_model.print_trainable_parameters()
    
    return lora_model

if __name__ == "__main__":
    # When this runs, it will print something like:
    # "trainable params: 2,500,000 || all params: 200,000,000 || trainable%: 1.25%"
    model = setup_lora_model()